In [ ]:
import os

import pandas as pd
from databricks.sdk.core import Config

from databricks import sql

config = Config()
connection = sql.connect(
    server_hostname=config.host.removeprefix("https://").rstrip("/"),
    http_path=os.environ.get("DATABRICKS_HTTP_PATH", "/sql/1.0/warehouses/6dc4b80538621b7b"),
    credentials_provider=lambda: config.authenticate,
    catalog="demo_redshift", schema="gold",
)


In [ ]:
query = """-- BI report: RFM segment sizes and value.
SELECT rfm_segment,
       COUNT(*)          AS customers,
       SUM(monetary)     AS total_monetary,
       CAST(AVG(frequency) AS BIGINT)    AS avg_frequency
FROM   gold.rfm_segments
GROUP  BY rfm_segment
ORDER  BY rfm_segment;
"""
rfm_segments = pd.read_sql_query(query, connection)
rfm_segments.head()


In [ ]:
query = """-- BI report: retention share per cohort x month offset.
SELECT cohort_month,
       month_offset,
       cohort_size,
       active_buyers,
       ROUND(100.0 * active_buyers / cohort_size, 2) AS retention_pct
FROM   gold.cohort_retention
ORDER  BY cohort_month, month_offset;
"""
cohort_retention = pd.read_sql_query(query, connection)
cohort_retention.head()


In [ ]:
connection.close()
